## Overview

The Burgers' equation is a paradigmatic nonlinear differential equation defined for 1D as 
$$
\frac{\partial u}{\partial t} = \nu \frac{\partial^2 u}{\partial x^2} - u \frac{\partial u}{\partial x} ,\quad u(x,0) = u_0(x),
$$
where $u(x,t)$ is the fluid velocity at poisition $x$ and time $t$, $\nu$ is the diffusion coefficient and $u_0(x)$ is the initial condition. 

Since the Burgers' equation is nonlinear, we embed it into a higher order linear system using the following transformations:
- Spatial discretization to transform the nonlinear PDE $\rightarrow$ nonlinear ODE.
- Carleman linearization to transform the finite, nonlinear ODE $\rightarrow$ infinite, linear ODE.
- Temporal discretization & truncation to transform the infinite, linear ODE $\rightarrow$ finite linear system of equations.

This results in the linear system $L^{(\text{e})}Y^{(\text{e})}=B^{(\text{e})}$ (see DH2026 for details), where we assume that $B^{(\text{e})}$ is normalized for simplicity. We will solve this system using the following methods:
- The Carleman linear Burgers' equation is efficiently loaded onto quantum hardware using an LCNU approach.
- The initial conditions are loaded using qiskit's `StatePreparation` routine.
- The linear system is solved using the Variational Quantum Linear Solver (VQLS; BP2023).

## Helpful References

- Gnanasekaran & Surana 2024. "Efficient variational quantum linear solver for structured sparse matrices" 
- Demirdjian & Hogancamp et al. 2026. "Quantum Data Loading for Carleman Linearized Systems: Application to the Lattice-Boltzmann Equation"
- Demirdjian & Quinn et al. 2026. "A Scalable Approach to Solve the Carleman Linearized Burgers' Equation on a Quantum Computer"
- Bravo-Prieto (2023). Variational quantum linear solver. Quantum, 7, 1188.

In [ ]:
#Libraries
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile, qpy
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_algorithms.optimizers import GradientDescent, CG, L_BFGS_B
import time
from qiskit.quantum_info import SparsePauliOp

#Custom libs
import Burgers_utils as bu
import circuit_utils as cu
import misc_utils as mu
import Carleman_Dilation

## Parameters

Define physical parameters for the Burgers' equation and algorithmic parameters for solving the linear system.

Key parameters:
<div align="center">

| Variable | Description |
| -------- | -------- | 
| $n_x$ | Number of discrete spatial points |
| $n_t$ | Number of discrete temporal points |
| $\alpha$ | Carleman truncation order |
| $n_\text{layer}$ | Number of ansatz layers |

</div>


Note that the size of the matrix $L^{(\text{e})}$ is $2n_t n_x^\alpha$. Therefore, we can only solve very small systems in this demo. However, as will be shown, we can create the circuits for extremely large systems. 

In [ ]:
#Parameters (each must be power of 2)
nx = 2**3 #number of spatial discretization points (>= 2**2)
nt = 2**2 #number of temporal discretization points
alpha = 2**1 #Carleman truncation order.
nqubit = int(np.log2(nt) + alpha*np.log2(nx) + 1 + 1) #Number of qubits to encode the Carleman matrix +1 for zero padding and +1 for ancilla

#Spatial parameters & initial condition
Length = 2*np.pi #units: [m]
dx   = Length/(nx-1)
ic_type = 'Gaussian' #'Gaussian','Impulse

#Temporal parameters
T = 5.       #Total duration, units: [s]
dt = T/nt

#Diffusion coefficient
nu = 0.25  #[m^2/s]

#Ansatz Parameters
lAnsatz = 'Sim18' #Sim9Mod','Sim18'
nlayer = 40 #Number of ansatz layers
if (lAnsatz=='Sim9Mod'):
    ntheta = nlayer * (nqubit-1)
elif (lAnsatz=='Sim18'):
    #ntheta = 3 * nlayer * (nqubit-1)
    ntheta = 2 * nlayer * (nqubit-1)
thetas = np.random.uniform(0,np.pi,ntheta)

#Optimizer parameters
opt = 'L_BFGS_B' #'GradientDescent', 'CG', 'L_BFGS_B'

#Options
lValidate = False #Validate the circuits for generating the Carlman matrix
lPauliCompare = False #Compare the cost of our approach with the Pauli Decomposition
lCreateCircs = False #Create and transpile circuits from scratch (slower), otherwise load from file (faster)
lTranspile = False #Transpile L^e circuits

#Set of parameters 
params = {'alpha':alpha,'nx':nx,'nt':nt,'nqubit':nqubit,
          'dt':dt,'dx':dx,'nu':nu,'nlayer':nlayer,
          'ntheta':ntheta,'Length':Length,'ic_type':ic_type,
          'opt':opt,'lTranspile':lTranspile}

print('Number of spatial grid points: nx = ',nx)
print('Number of temporal grid points: nt = ',nt)
print('Carleman truncation order: α = ', alpha)
print('Dimension of linear system: ', 2**(nqubit-1))
print('Number of ansatz layers: nlayer = ',nlayer)
print('Number of variational parameters: nθ = ',ntheta)
print('Initial condition type: ic_type = ',ic_type)
print('optimizer: opt = ',opt)

## Create Circuits

Consider the LCNU $L^{(\text{e})} = \sum_{l=0}^{N_s} c_l L_l$, for complex coefficients $c_l$ and specific non-unitaries $L_l$.

Next, define $V(\vec{\theta})$ as an ansatz, which is a parameterizable circuit. 

Create the following:

<div align="center">

| Variable | Description |
| -------- | -------- | 
| `qc_carl` | List of circuits for all $L_l$ terms |
| `coeffs` | List of the $c_l$ coefficients |
| `qc_init` | Circuit for $B^{(\text{e})}$ |
| `qc_anz` | Ansatz circuit for $V(\vec{\theta})$ |

</div>

Following DQ2026, the number of terms in the LCNU of $L^{(\text{e})}$ is exactly $N_s=\frac{1}{2}(9\alpha^2 + \alpha + 8)$.
For $\alpha=2$ this gives $N_s=23$.

If `lValidate=True`, then we validate $L^{(\text{e})}$ from the circuits against its analytical form.


In [ ]:
#Create the circuits and coefficients for the Carleman matrix
qc_carl,coeffs = bu.create_circs(params)
Ns = len(coeffs) #number of circuits
params['Ns'] = Ns

#Plot spectrum
fig, ax = plt.subplots()
p1 = ax.plot(sorted(np.abs(coeffs)),'-o',c='black')
plt.title('Coefficient Spectrum',fontsize=14)
plt.xlabel('Sorted Coefficient',fontsize=14)
plt.ylabel('Coefficient Magnitude',fontsize=14)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
plt.savefig("Coefficients.png")

#Create Ansatz  
if (lAnsatz=='Sim9Mod'):
    qc_anz = cu.Ansatz_Sim9_Modified(params)
elif (lAnsatz=='Sim18'):
    qc_anz = cu.Ansatz_Sim18(params)

#Initial condition circuit
Be = mu.normalized_initial_condition(params) #normalized vector
Be = np.asarray(Be, dtype=complex)
stateprep = StatePreparation(Be)
qc_init = QuantumCircuit(nqubit-1)
qc_init.append(stateprep, list(range(0,nqubit-1)))
qc_init = qc_init.decompose(reps=5)

#Ansatz transpilation 
simulator = AerSimulator()
qc_anz = transpile(qc_anz, simulator, optimization_level=3)

#Validate the circuits and coefficients by comparing the Carleman matrix generated from the circuits 
#with the Carleman matrix generated from matrices
if (nqubit <= 11 and lValidate):
    Le_class,Le_circ = bu.validate_CarlemanDilated_Matrix(qc_carl,coeffs,params)

    nA = np.sum([nx**i for i in range(1,alpha+1)])
    #Plot exact approach versus circuit approach
    fig, ax = plt.subplots(1,2)
    p1 = ax[0].spy(Le_class, aspect = 'auto', markersize=2, alpha = 1.)
    p2 = ax[1].spy(Le_circ, aspect = 'auto', markersize=2, alpha = 1.)# Draw a horizontal line at y = 3
    for axi in ax:
        axi.set_aspect('equal')
    ax[0].set_title(r'$L^{(\text{e})}$ from Analytic Form')
    ax[1].set_title(r'$L^{(\text{e})}$ from Circuits')
    plt.savefig("TestMatrix.png")

## VQLS

The VQLS cost function is 
$$
C(\vec{\theta}) = \frac{1}{2} \left( 1 - \frac{1}{q}\frac{\sum_{k=0}^{q-1} \sum_{i,j=0}^{N_s-1} c_ic_j^*\delta_{i,j,k}}{\sum_{i,j=0}^{N_s-1} c_ic_j^*\beta_{i,j}}  \right),
\\[10pt]
\delta_{i,j,k} = \bra{V(\vec{\theta})} L_j^* U_b Z_k U_b^* L_i \ket{V(\vec{\theta})}, 
\\[10pt]
\beta_{i,j} = \bra{V(\vec{\theta})} L_j^* L_i \ket{V(\vec{\theta})},
$$
where each term is defined above and $Z_k$ is the Pauli-Z gate applied to the $k\text{th}$ qubit.

### Store observables in cache

There are $qN_s^2\times$ $\delta_{i,j,k}$ circuits and $N_s^2\times$ $\beta_{i,j}$ circuits, resulting in a total of $N_s^2(q+1)$ circuits.

Since this is slow to run, we will store the following fixed observables in cache:

<div align="center">

| Variable | Description |
| -------- | -------- | 
| `obs_delta` | Defined as $L_j^* U_b Z_k U_b^* L_i$, which are the $\delta_{i,j,k}$ observables |
| `obs_beta` | Defined as $L_j^* L_i$, which are the $\beta_{i,j}$ observables |

</div>

In [ ]:
#U_b Z_k U_b^dagger
UZUd = bu.extract_UZUd(qc_init,params)

#Store A_j^dagger U_bZ_kU_b^dagger A_i in cache
Aj = bu.extract_Aj_from_Uj(qc_carl,params)

obs_beta = 0
obs_delta = 0
coeffs_ij = []
coeffs_ijk = []
for i in range(0,Ns):
    for j in range(0,Ns):
        c = coeffs[i] * np.conjugate(coeffs[j])
        coeffs_ij.append(c) #ij
        obs_beta += c * Aj[j].conj().T @ Aj[i]
        for k in range(0,nqubit-1):
            coeffs_ijk.append(c) #ijk
            obs_delta += c * Aj[j].conj().T @ UZUd[k] @ Aj[i]
del([c])

### Run VQLS

Minimize the above cost function taking matrix products, which is mathematically equivalent to running the circuits but much faster.

Code will take ~30s with $n_x=4$, $n_t=2$, $\alpha=2$.

In [ ]:


#Calculate cost function using cached matrices
backend = AerSimulator(method="statevector")
def calculate_cost_function(thetas):

    #update the variational parameters
    qc_anz0 = qc_anz.assign_parameters(thetas)
    qc_anz0.save_statevector()
    sv_anz = backend.run(qc_anz0).result().get_statevector().data

    #\beta_ij and \delta_ijk terms
    denom = np.vdot(sv_anz, obs_beta @ sv_anz)
    numer = np.vdot(sv_anz, obs_delta @ sv_anz)
    cost = 0.5*(1 - 1/(nqubit-1) * numer.real/denom.real)
    return(cost)

#callback to store the cost function at each iteration
cost_history = []
x_history = []
def store_intermediate_result(xk):
    x_history.append(xk.copy())
    val = calculate_cost_function(xk)
    cost_history.append(val)
    print(val)

#Optimizer parameters
nitr = 250
maxfun = 1e6 #15000
tol = 1e-6
gtol = 1e-4
eps = 1e-8
bounds = [(0, 2 * np.pi)] * params['ntheta']

if params['opt']=='CG':
    optimizer = CG(maxiter=nitr, tol=tol, gtol=gtol, callback=store_intermediate_result)
    theta_opt = optimizer.minimize(fun=calculate_cost_function, x0=thetas)
elif params['opt']=='L_BFGS_B':
    optimizer = L_BFGS_B(maxfun=maxfun,maxiter=nitr, eps=eps, ftol=tol, callback=store_intermediate_result)
    theta_opt = optimizer.minimize(fun=calculate_cost_function,x0=thetas,bounds=bounds)
elif params['opt']=='GradientDescent':
    optimizer = GradientDescent(maxiter=nitr, tol=tol, callback=store_intermediate_result)
    theta_opt = optimizer.minimize(fun=calculate_cost_function, x0=thetas)


print(theta_opt)


In [ ]:
#Plot the cost function history
plt.plot(cost_history, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Cost function per iteration")
plt.yscale('log')
plt.show()


In [ ]:

#Classical solution
Le_class = Carleman_Dilation.Carleman_Dilation_Matrix(params)
x_class = mu.solve_sys(Le_class,Be,params)

#Quantum solution obtained by evaluating the ansatz with optimized parameters
qc_anz0 = qc_anz.assign_parameters(theta_opt.x)
qc_anz0.save_statevector()
backend = AerSimulator(method="statevector")
job = backend.run(qc_anz0)
result = job.result()
sv_anz = result.get_statevector().data.real

#Extract quantum solution at each time step
x_qutm = np.zeros((nt,nx))
delta0 = np.sum([int(nx**j) for j in range(1,alpha+1)])
for i in range(nt):
    offset = int((i+1)*2*nx**alpha - delta0)
    x_qutm[i,:] = sv_anz[offset:nx+offset]
factor = 1 if np.sqrt(np.mean((x_qutm-x_class)**2)) < np.sqrt(np.mean((-x_qutm-x_class)**2)) else -1 #Sometimes solution is inverterd
x_qutm = factor * x_qutm

#Plot real solution with quantum solution
fig, ax = plt.subplots(1,2)
for i in range(0,nt):
    p1 = ax[0].plot(x_class[i,:],'-o',label=rf't={i}$\Delta$t')
    p2 = ax[1].plot(x_qutm[i,:],'-o')
ax[0].set_title("Classical Solution", fontsize=14)
ax[0].set_xlabel("Spatial Grid", fontsize=14)
ax[0].set_ylabel(r"Fluid Velocity $u(x,t)$", fontsize=14)
ax[1].set_title("Quantum Solution", fontsize=14)
ax[1].set_xlabel("Spatial Grid", fontsize=14)
ax[0].legend(loc='lower center')
plt.savefig("Solution.png")


## How does our LCNU method compare with a Pauli decomposition? 

To answer this question, we will compare the number of terms (NoT) in the the LCNU case with that of the Pauli decomposition case.

The NoT for the LCNU method was previously stated to be $N_s=\frac{1}{2}(9\alpha^2 + \alpha + 8)$. 

The NoT for the Pauli decomposition will be numerically calculated by 
- First, analytically forming the $L^{(\text{e})}$ matrix,
- Then, using qiskit's `SparsePauliOp.from_operator` function to decompose it using the Pauli decomposition. 

This process is repeated for different values of $n_x$ and $n_t$ with fixed $\alpha$. Ideally, we would also vary $\alpha$, but the system grows too rapidly for the qiskit's `SparsePauliOp.from_operator` to process. 



In [ ]:
#Pauli
NoT_paul = [] #Number of terms in Pauli decomposition
NoT_lcnu = []
Le_dim = [] #Dimension of L^e matrix
params0 = params.copy()
for i in range(1,6):
    j = i if i<5 else 4
    params0['nt'] = 2**i
    params0['nx'] = 2**j
    Le = Carleman_Dilation.Carleman_Dilation_Matrix(params0)
    NoT_paul.append(SparsePauliOp.from_operator(Le).size)  
    Le_dim.append(2 * params0['nt'] * params0['nx']**params0['alpha'] ) #Dimension of Carleman matrix

    #Number of terms in LCNU
    NoT_lcnu.append((9*params0['alpha']**2 + params['alpha'] + 8)/2)

#Plots
fig, ax = plt.subplots()
ax.plot(Le_dim,NoT_paul,'-o',c='black',label='Pauli')
ax.plot(Le_dim,NoT_lcnu,'-^',c='blue',label='LCNU')

plt.xlabel("Carleman Matrix Dimension",fontsize=14)
plt.ylabel("Number of Terms in Decomposition",fontsize=14)
plt.title("LCNU vs Pauli Decomposition",fontsize=14)
ax.legend(loc='upper left')
ax.set_yscale('log') 
ax.set_xscale('log') 
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
plt.savefig("PauliDecomp.png")

### Scale up to rediculous dimension

In DQ2026, we showed that the LCNU for the 1D Burgers' equation scales efficiently. As previously mentioned, the NoT scales like $\mathcal{O}(\alpha^2)$. But, what are the circuit depths for each term in the LCNU? 

To answer this question, we create the exact circuits and then use qiskit to measure their depths. This is done in two parts:
- Fix $\alpha=4$ and create the circuits as $n_x$ and $n_t$ are varied from $2^{10}$ to $2^{50}\approx 10^{15}$.
- Fix $n_x=n_t=2^{50}$ and create the circuits as $\alpha$ is varied from $2$ to $8$.

This will take ~30s to run.

In [ ]:
#New parameters at rediculous scale
params_rediculous = params.copy()
params_rediculous['nx'] = 2**50
params_rediculous['nt'] = 2**50
params_rediculous['alpha'] = 2**2
params_rediculous['nqubit'] = int(np.log2(params_rediculous['nt']) + params_rediculous['alpha']*np.log2(params_rediculous['nx']) + 1 + 1)
print(params_rediculous)

# #Create the circuits and coefficients for the Carleman matrix
# qc_carl,coeffs = bu.create_circs(params_rediculous)
# Ns = len(coeffs) #number of circuits
# params_rediculous['Ns'] = Ns
# print(Ns)

#Get the circuit depths as a function of nx and nt for fixed alpha
params_rediculous = params.copy()
alpha_fixed = 2**2
params_rediculous['alpha'] = alpha_fixed
max_depth_nxnt = []
matrix_dim_nxnt = []
for i in range(10,60,10):
    print('Fixed alpha, working on ',i)
    params_rediculous['nx'] = 2**i
    params_rediculous['nt'] = 2**i
    params_rediculous['nqubit'] = int(np.log2(params_rediculous['nt']) + params_rediculous['alpha']*np.log2(params_rediculous['nx']) + 1 + 1)
    circs,coeffs = bu.create_circs(params_rediculous)
    max_depth_nxnt.append(np.max([x.depth() for x in circs])) #Maximum circuit depth for each configuration
    matrix_dim_nxnt.append(2**(params_rediculous['nqubit']-1)) #Matrix dimension of the associated Carleman linearized system
    del([circs])

#Get the circuit depths as a function of alpha for fixed nx and nt
nxnt_fixed = 2**50
params_rediculous['nx'] = nxnt_fixed
params_rediculous['nt'] = nxnt_fixed
max_depth_alpha = []
matrix_dim_alpha = []
for i in range(1,4):
    print('Fixed nx and nt, working on ',i)
    params_rediculous['alpha'] = 2**i
    params_rediculous['nqubit'] = int(np.log2(params_rediculous['nt']) + params_rediculous['alpha']*np.log2(params_rediculous['nx']) + 1 + 1)
    circs,coeffs = bu.create_circs(params_rediculous)
    max_depth_alpha.append(np.max([x.depth() for x in circs])) #Maximum circuit depth for each configuration
    matrix_dim_alpha.append(2**(params_rediculous['nqubit']-1)) #Matrix dimension of the associated Carleman linearized system
    del([circs])

In [ ]:
#Plot circuit depths
fig, ax = plt.subplots(1,2,figsize=(8,4))
ax[0].plot(matrix_dim_nxnt,max_depth_nxnt,'-o',c='black',label='Pauli')
ax[1].plot(matrix_dim_alpha,max_depth_alpha,'-o',c='black',label='Pauli')

ax[0].set_xlabel(r"$L^{(\text{e})}$ Dimension",fontsize=14)
ax[0].set_ylabel(r"Circ Depth, Variable $n_x,n_t$",fontsize=14)
ax[0].set_title(rf'Fixed $\alpha=${alpha_fixed}',fontsize=14)
ax[0].set_xscale('log') 
ax[0].tick_params(axis='x', labelsize=12)
ax[0].tick_params(axis='y', labelsize=12)

ax[1].set_xlabel(r"$L^{(\text{e})}$ Dimension",fontsize=14)
ax[1].set_ylabel(r"Circ Depth, Variable $\alpha$",fontsize=14)
ax[1].set_title(rf'Fixed $n_x=n_t=2^{{{int(np.log2(nxnt_fixed))}}}$',fontsize=14)
ax[1].set_xscale('log') 
ax[1].tick_params(axis='x', labelsize=12)
ax[1].tick_params(axis='y', labelsize=12)
plt.tight_layout() 
plt.savefig("Max_circ_depth.png")


## Conclusions

In this tutorial, we have done the following:
- Created LCNU circuits for the 1D Carleman linearized Burgers' equation,
- Solved the Carleman linearized system using VQLS,
- Shown that the LCNU decomposition outperforms the Pauli decomposition,
- Proven that the LCNU approach can efficiently load the 1D Carleman Burgers' equation as $n_x$ and $n_t$ are scaled up exponentially. 

Final note: while we demonstrated an efficient loading method, it is important to point out that this does *not* mean that we have a full end-to-end quantum algorithm to obtain a solution. To do that, we would need to not only do preconditioning on $L^\text{(e)}$, but we would also need to determine what information to extract from the final state. 